#### Proprocess

In [13]:
import pandas as pd

# load the three umls files
root_path = r'umls_datasets'
data_src = 'SCT'
umls_def = pd.read_csv(f'{root_path}/{data_src}_2024AB_DEF.csv')
umls_doc = pd.read_csv(f'{root_path}/{data_src}_2024AB_DOC.csv')
umls_rela = pd.read_csv(f'{root_path}/{data_src}_2024AB_RELA.csv').drop_duplicates()
umls_doc['EXPL'] = umls_doc['EXPL'].str.lower()
umls_doc['EXPL'] = umls_doc['EXPL'].str.replace('_', ' ')

# combine the names and definitions into the relations
umls_rela = pd.merge(umls_rela, umls_def, left_on='CUI1', right_on='cui', how='left')
umls_rela = pd.merge(umls_rela, umls_def, left_on='CUI2', right_on='cui', how='left')
umls_rela = pd.merge(umls_rela, umls_doc, left_on='REL', right_on='VALUE', how='left')

# use names for entities
columns = {'name_x': 'head', 'EXPL': 'relation', 'name_y': 'tail'}
umls_rela_name = umls_rela[['name_x', 'EXPL', 'name_y']].dropna().rename(columns=columns).reset_index(drop=True)

# # use defs for entities
# columns = {'definition_x': 'head', 'EXPL': 'relation', 'definition_y': 'tail'}
# umls_rela_def = umls_rela[['definition_x', 'EXPL', 'definition_y']].dropna().rename(columns=columns).reset_index(drop=True)

In [ ]:
umls_rela_name.to_csv(rf'{root_path}/{data_src}_2024AB_RELA_NAME.csv', index=False)

#### Group by Relations

In [14]:
umls_rela = umls_rela_name
# Group by relation
umls_rela_group = umls_rela.groupby('relation').size().reset_index(name='Count').sort_values(by='Count', ascending=False, ignore_index=True)
umls_rela_group

,relation,Count
0,inverse is a,594753
1,is a,594753
2,has finding site,247424
3,finding site of,247424
4,has associated morphology,147701
...,...,...
130,has temporal context,1
131,temporal context of,1
132,plays role,1
133,approach of,1


#### Filter the Switchable Relations

In [15]:
switch_sets = [{'has part', 'part of'}, {'form of', 'has form'}, {'has exam', 'is exam for'}, {'has stage', 'stage of'}, {'has count', 'count of'},
               {'occurs in', 'has occurrence'},
               {'formed by', 'forms'}, {'treats', 'treated by'}, {'plays role', 'role played by'}, {'receives input from', 'sends output to'}, {'before', 'after'},
               {'gene is biomarker of', 'disease is marked by gene'}, {'organism has gene', 'is stage of disease'}]

switch_lists = [['is a', 'inverse is a'], ['see from', 'see'], ['used for', 'use'], ['due to', 'cause of'],
                ['used by', 'uses'], ['forms', 'formed by'], ['disease is marked by gene', 'gene is biomarker of']]

def same_consecutive_chars(str1, str2, window_size):
    # Loop over each character in the first string
    for i in range(len(str1) - window_size + 1):  # The window size is 5
        substring = str1[i:i + window_size]  # Get the 5-character substring
        if substring in str2:  # Check if this substring exists in the second string
            return True
    return False

def return_has_or_shorter(str1, str2):
    if 'has' in str1:
        return [str1]
    if 'has' in str2:
        return [str2]
    return [str1] if len(str1) < len(str2) else [str2]

def choose_rel(rels):
    if len(rels) == 2 and same_consecutive_chars(rels[0], rels[1], window_size=5):
        return return_has_or_shorter(rels[0], rels[1])

    elif len(rels) == 2:
        if {rels[0], rels[1]} in switch_sets:
            return return_has_or_shorter(rels[0], rels[1])
        elif [rels[0], rels[1]] in switch_lists:
            return [rels[0]]
        elif [rels[1], rels[0]] in switch_lists:
            return [rels[1]]
        else:
            return [rels[0], rels[1]]

    elif len(rels) > 2:
        result = []
        # Compare each pair of relations
        for rel in rels:
            if rel not in result:
                is_unique = True
                for selected_rel in result:
                    if {rel, selected_rel} in switch_sets:
                        is_unique = False
                        break
                    elif same_consecutive_chars(rel, selected_rel, 6):
                        is_unique = False
                        break
                if is_unique:
                    result.append(rel)
        return result
    else:
        return rels

# Group relations by Count
umls_rela_count_groups = umls_rela_group.groupby('Count')['relation'].apply(list).to_dict()
# Filter the relations
filtered_relations = []
for rels in umls_rela_count_groups.values():
    filtered_relations.extend(choose_rel(rels))

len(filtered_relations)

70

In [4]:
a = {cnt: choose_rel(rels) for cnt, rels in umls_rela_count_groups.items()}
for cnt, rels in a.items():
    if len(rels) >= 2:
        print(cnt, rels)

1 ['procedure has target anatomy', 'treated by', 'allele absent from wild-type chromosomal location', 'induced by', 'internal to', 'direct cell shape of', 'has germ origin', 'chemical or drug has physiologic effect', 'reformulation of']
2 ['has segmental composition', 'biological process has result anatomy', 'has inherent 3-d shape', 'negatively regulates']
3 ['contraindicated physiologic effect of', 'has multi-level category', 'has full grown phenotype', 'expanded form of']
5 ['right medial to', 'is component of chemotherapy regimen', 'has ingredient', 'left lateral to']
6 ['chemical or drug is metabolized by enzyme', 'merges with', 'contraindicated class of']
7 ['has adherent', 'has active ingredient', 'has fusion']
13 ['abnormal cell affected by chemical or drug', 'cell type is associated with eo disease']
15 ['forms', 'has related developmental entity']
20 ['has component', 'inferomedial to', 'superolateral to']
22 ['adjacent to', 'is mechanism of action of chemical or drug']
26 ['

In [16]:
# Filter the umls_rela
relations_to_remove = ['was a', 'mapped to', 'replaces', 'has tradename', 'tradename of', 'has print name', 'has suffix',
                       'lab number', 'has lab number', 'has loinc number', 'loinc number of', 'has version', 'has scale', 'classifies class code', 'class code classified by',
                       'has pcdc hl authorized value',
                       'has pcdc ews authorized value',
                       'has pcdc os authorized value',
                       'has pcdc aml authorized value',
                       'has pcdc all authorized value',
                       'has pcdc gct authorized value',
                       'is pcdc gct authorized value for variable',
                       'has dipg dmg authorized value', 'has acc-aha sars2 authorized value', 'has seronet authorized value',
                       'has gdc value',
                       'has ctdc value', 'is ctdc value of',
                       'has icdc value', 'icdc value of',
                       'has ooro pc value',
                       'value set is paired with',
                       'has ingredient', 'has active ingredient', 'has inactive ingredient',
                       'has precise ingredient', 'has precise active ingredient',
                       'has count', 'concept in subset']
filtered_umls_rela = umls_rela[umls_rela['relation'].isin(filtered_relations)]
filtered_umls_rela = filtered_umls_rela[~filtered_umls_rela['relation'].isin(relations_to_remove)]
filtered_umls_rela

,head,relation,tail
1,"1,4-alpha-Glucan branching enzyme (substance)",same as,"1,4-alpha-Glucan 6alpha-glucosyltransferase (s..."
7,2-Aminoadipic acid,same as,2-Aminoadipate
16,Beta alanine (substance),is a,N-carbamoyl-beta-alanine (substance)
17,Beta alanine (substance),is a,3-Ureidopropionate
30,6-Aminocaproic acid,has causative agent,Poisoning caused by aminocaproic acid (disorder)
...,...,...,...
2485553,Recurrent hemarthrosis of joint,is a,Recurrent hemarthrosis of joint of right shoul...
2485554,Recurrent hemarthrosis of joint,is a,Recurrent hemarthrosis of left elbow joint (di...
2485555,Recurrent hemarthrosis of joint,is a,Recurrent hemarthrosis of right elbow joint
2485560,Finding related to digital literacy,is a,Adequate basic digital literacy skills


In [17]:
filtered_umls_rela_group = filtered_umls_rela.groupby('relation').size().reset_index(name='Count').sort_values(by='Count', ascending=False, ignore_index=True)
filtered_umls_rela_group

,relation,Count
0,is a,594753
1,has finding site,247424
2,has associated morphology,147701
3,has part,52237
4,possibly equivalent to,45036
...,...,...
61,has focus,4
62,has temporal context,1
63,role played by,1
64,has approach,1


In [18]:
filtered_umls_rela.to_csv(rf'{root_path}/{data_src}_2024AB_RELA_NAME.csv', index=False)

#### Combine Triplets

In [23]:
import pandas as pd

root_path = r'umls_datasets'
data_src = 'UMLS_new'
umls_rela = pd.read_csv(rf'{root_path}/{data_src}_2024AB_RELA_NAME.csv')
umls_rela['triplet'] = umls_rela.apply(lambda row: f"{row['head'].strip()} {row['relation'].strip()} {row['tail'].strip()}", axis=1)
umls_rela['id'] = umls_rela.index.map(lambda x: f'D{x}')
umls_rela

,head,relation,tail,triplet,id
0,beta alanine,used for,"AMINO ACIDS, SOURCE UNSPECIFIED","beta alanine used for AMINO ACIDS, SOURCE UNSP...",D0
1,beta alanine,has structural class,"AMINO ACIDS, SOURCE UNSPECIFIED","beta alanine has structural class AMINO ACIDS,...",D1
2,beta alanine,has contraindicated drug,"Allergies, Drug",beta alanine has contraindicated drug Allergie...,D2
3,4-Hydroxyphenylpyruvate Dioxygenase,used for,oxygenase,4-Hydroxyphenylpyruvate Dioxygenase used for o...,D3
4,Pyrimidine 5'-Nucleotidase,gene product has chemical classification,"NT5C3A protein, human",Pyrimidine 5'-Nucleotidase gene product has ch...,D4
...,...,...,...,...,...
1063427,Cortical thickening of the fibula,is a,Thickened femoral cortex,Cortical thickening of the fibula is a Thicken...,D1063427
1063428,Positive respiratory tract nucleic acid pathog...,is a,Positive respiratory tract SARS-CoV-2 coronavi...,Positive respiratory tract nucleic acid pathog...,D1063428
1063429,Biospecimen phenotypic feature,is a,Abnormal lymph-node biospecimen,Biospecimen phenotypic feature is a Abnormal l...,D1063429
1063430,Biospecimen phenotypic feature,is a,Marker expression,Biospecimen phenotypic feature is a Marker exp...,D1063430


In [20]:
umls_rela[['id', 'triplet']].to_csv(rf'{root_path}/{data_src}_2024AB_RELA_NAME_triplet.csv', index=False)

#### Statistics

In [1]:
import pandas as pd
from transformers import AutoTokenizer

root_path = r'umls_datasets'
data_src = 'UMLS_new'
umls_rela = pd.read_csv(f'{root_path}/pmc_oa_captions.csv')
tokenizer = AutoTokenizer.from_pretrained('microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext')
umls_rela['num_tokens'] = umls_rela['triplet'].apply(lambda x: len(tokenizer.tokenize(x)))
umls_rela_g = umls_rela.groupby('num_tokens').size().reset_index(name='Count').sort_values(by='Count', ascending=False, ignore_index=True)
umls_rela_g

,num_tokens,Count
0,21,2895
1,19,2828
2,23,2821
3,22,2821
4,26,2802
...,...,...
1022,964,1
1023,965,1
1024,968,1
1025,654,1


In [12]:
# Compute the number of rows where num_tokens < i
total = umls_rela_g['Count'].sum()
count_less_than = umls_rela_g[umls_rela_g['num_tokens'] <= 512]['Count'].sum()
# Compute the percentage
total, count_less_than, (count_less_than / total) * 100

(378694, 375603, 99.1837737064754)

In [52]:
test_clean = pd.read_csv(r'E:\PycharmProjects\SOURCE\PMC-VQA\test_clean.csv')
test_clean['num_q_tokens'] = test_clean['Question'].apply(lambda x: len(tokenizer.tokenize(x)))
test_clean_g = test_clean.groupby('num_q_tokens').size().reset_index(name='Count').sort_values(by='num_q_tokens', ascending=False, ignore_index=True)
test_clean_g

,num_q_tokens,Count
0,34,1
1,32,1
2,31,1
3,29,2
4,26,3
5,24,5
6,23,5
7,22,8
8,21,13
9,20,25


In [68]:
# Compute the number of rows where num_tokens < 512
total = test_clean_g['Count'].sum()
count_less_than = test_clean_g[test_clean_g['num_q_tokens'] <= 30]['Count'].sum()
# Compute the percentage
total, count_less_than, (count_less_than / total) * 100

(2000, 1997, 99.85000000000001)